In [1]:
import malariagen_data

import pandas as pd
import numpy as np

ag3 = malariagen_data.Ag3()

ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : None
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.10.0
Client location                       : Alberta, Canada
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.10.0/Ag3.html

In [12]:
sample = "1244-VO-GH-YAWSON-VMF00051"

ds_snps = ag3.snp_calls(
    region="3L",
    sample_sets=sample,
    site_mask="gamb_colu_arab"
)
ds_snps

<xarray.Dataset> Size: 293GB
Dimensions:                             (variants: 25869385, alleles: 4,
                                         samples: 666, ploidy: 2)
Coordinates:
    variant_position                    (variants) int32 103MB dask.array<chunksize=(85223,), meta=np.ndarray>
    variant_contig                      (variants) uint8 26MB dask.array<chunksize=(85223,), meta=np.ndarray>
    sample_id                           (samples) <U24 64kB dask.array<chunksize=(666,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele                      (variants, alleles) |S1 103MB dask.array<chunksize=(85223, 4), meta=np.ndarray>
    variant_filter_pass_gamb_colu_arab  (variants) bool 26MB dask.array<chunksize=(60227,), meta=np.ndarray>
    variant_filter_pass_gamb_colu       (variants) bool 26MB dask.array<chunksize=(60227,), meta=np.ndarray>
    variant_filter_pass_arab            (variants) bool 26MB dask.array<chunksize=(60227,), meta=np.ndarray>
    call_genotype                       (variants, samples, ploidy) int8 34GB dask.array<chunksize=(60227, 50, 2), meta=np.ndarray>
    call_GQ                             (variants, samples) int8 17GB dask.array<chunksize=(60227, 50), meta=np.ndarray>
    call_MQ                             (variants, samples) float32 69GB dask.array<chunksize=(60227, 50), meta=np.ndarray>
    call_AD                             (variants, samples, alleles) int16 138GB dask.array<chunksize=(60227, 50, 4), meta=np.ndarray>
    call_genotype_mask                  (variants, samples, ploidy) bool 34GB dask.array<chunksize=(60227, 50, 2), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

In [3]:
num_samples = ds_snps.sizes['samples']
ploidy = ds_snps.sizes['ploidy']

ac = ag3.snp_allele_counts(region="3L", sample_sets=sample, site_mask="gamb_colu_arab")
locus_counts = np.sum(ac, axis=1)
no_missing_data = locus_counts == num_samples * ploidy
np.count_nonzero(no_missing_data)

Compute SNP allele counts:   0%|          | 0/2585 [00:00<?, ?it/s]

21655536

In [4]:
base_counts = np.count_nonzero(ac, axis=1)
print(base_counts)
at_least_two = np.nonzero(base_counts >= 2)[0]
print(np.count_nonzero(base_counts == 3))
print(np.count_nonzero(base_counts == 4))
print(at_least_two.shape)

[1 1 1 ... 1 1 1]
1307959
93299
(8329950,)


In [5]:
site_filter = (base_counts >= 2) & no_missing_data
np.count_nonzero(site_filter)
site_indices = np.nonzero(site_filter)[0]

In [6]:
samples = 100_000

rng = np.random.default_rng(42)

random_indices = rng.choice(site_indices, size=samples, replace=False)

random_indices.sort()

In [ ]:
smaller_ds = ds_snps.isel(variants=random_indices)

alleles = smaller_ds["variant_allele"].values
genotypes = smaller_ds["call_genotype"].values

# optional
np.save("npy/genotypes-3L-100k-seg-no-missing.npy", genotypes)

# print(sample_ids.shape)
print(alleles.shape)
print(genotypes.shape)

In [3]:
genotypes = np.load("../npy/genotypes-3L-100k-seg-no-missing.npy")

In [5]:
print(np.count_nonzero(genotypes < 0))

0


In [16]:
import kestrel

# Nothing is missing, so this will be valid

view = genotypes.view(np.uint8)
print(view.dtype)
print(view.shape)

jacquard = kestrel.relatedness_coefficients_gt(view)

sample_ids = ds_snps["sample_id"].values

uint8
(100000, 666, 2)
AVX-512 detected, using SIMD intrinsics
Calculating Jacquard coefficients for 222111 pairs using 100000 variants
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% ✔ Done!                       


In [17]:
print(jacquard.shape)

(666, 666, 9)


In [18]:
g_a = 0
for s in list(sample_ids):
    if "VBS19441" in s:
        break
    g_a += 1

print(g_a)

g_b = 0
for s in list(sample_ids):
    if "VBS19464" in s:
        break
    g_b += 1
print(g_b)


243
257


In [19]:
c_a = 0
for s in list(sample_ids):
    if "VBS19775" in s:
        break
    c_a += 1

print(c_a)

c_b = 0
for s in list(sample_ids):
    if "VBS19795" in s:
        break
    c_b += 1
print(c_b)


501
520


In [20]:
u_a = 0
for s in list(sample_ids):
    if "VBS19051" in s:
        break
    u_a += 1

print(u_a)

u_b = 0
for s in list(sample_ids):
    if "VBS19052" in s:
        break
    u_b += 1
print(u_b)

0
1


In [23]:
kv = np.array([1.0, 0.0, 0.5, 0.0, 0.5, 0.0, 0.5, 0.25, 0.0])
kinship = jacquard @ kv

In [24]:
print("Most related gambiae", kinship[g_a, g_b])
print("Most related coluzzi", kinship[c_a, c_b])
print("Unrelated", kinship[u_a, u_b])

Most related gambiae 0.4987579130793315
Most related coluzzi 0.5074073729694567
Unrelated 0.08180264190685263


In [14]:
print("Most related gambiae", kinship[g_a, g_b])
print("Most related coluzzi", kinship[c_a, c_b])
print("Unrelated", kinship[u_a, u_b])

Most related gambiae 0.5036185054399601
Most related coluzzi 0.5074073729840259
Unrelated 0.0818026418657589


In [25]:
import matplotlib.pyplot as plt

plt.hist(kinship.flatten())
plt.title("Kinship Coefficients for Mosquitoes")
plt.ylabel("Number of Samples")
plt.xlabel("Kinship")

inbreeding = 2*np.diag(kinship) - 1


ModuleNotFoundError: No module named 'matplotlib'

In [16]:
import networkx as nx

import plotly.graph_objects as go

threshold = 0.25

G = nx.Graph()
for name in sample_ids:
    G.add_node(name)

for i in range(len(sample_ids)):
    for j in range(i + 1, len(sample_ids)):
        if kinship[i, j] > threshold:
            G.add_edge(sample_ids[i], sample_ids[j], weight=kinship[i, j])

pos = nx.spring_layout(G, seed=42)

edge_x = []
edge_y = []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=1.5, color='#888'),
    hoverinfo='none',
    mode='lines'
)

node_x = []
node_y = []
node_text = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(f"Sample: {node}<br>Connections: {len(list(G.neighbors(node)))}")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    text=list(G.nodes()),
    textposition="top center",
    hoverinfo='text',
    hovertext=node_text,
    marker=dict(
        size=10,
        color=inbreeding,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Inbreeding', thickness=15),
        line=dict(width=2, color='DarkSlateGrey')
    )
)

fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title=f"Relatedness Graph (Kinship > {threshold})",
        showlegend=False,
        hovermode='closest',
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
)

fig.show()

# histogram for kinship
# sample location, lat/long when hover, date, country, admin region
# plot kinship vs geographic distance (nice to have)